# Buổi 17 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `arima.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Mô phỏng và ACF/PACF (mục 4.1)

Đoán bậc từng mô hình trước khi nhìn tên.

In [ ]:
%matplotlib inline
import warnings

import arima as ar
import numpy as np

warnings.simplefilter("ignore")
for ten, tham_so in {"AR(2)": dict(ar=[0.6, 0.3]), "MA(1)": dict(ma=[0.8]), "ARMA(1,1)": dict(ar=[0.7], ma=[0.4])}.items():
    y = ar.mo_phong(n=500, seed=1, **tham_so)
    print(ten, "| ngưỡng ±", round(1.96 / np.sqrt(len(y)), 3))
    print(ar.acf_pacf(y, 6).round(2).T.to_string())

## Bước 2 — Sản lượng công nghiệp (mục 4.2, 4.4)

Sửa `kiem_phan_du` rồi chạy lại ô này.

In [ ]:
ip = np.log(ar.doc_g17()[:"2019-12"].to_numpy()) * 100
print(ar.acf_pacf(np.diff(ip), 6).round(2).T.to_string())
print(ar.bang_ung_vien(ip, [(1, 1, 0), (4, 1, 0), (1, 1, 1), (2, 1, 2)]).round(3).to_string(index=False))

## Bước 3 — Chuỗi du lịch T33 (mục 4.3)

Sửa mặc định của `mo_hinh_tu_chon`, rồi `MUA_VU_ARIMA`, chạy lại ô này sau mỗi lần sửa.

In [ ]:
t33 = np.log(ar.doc_tourism()["T33"][:-ar.TAM])
mh = ar.mo_hinh_tu_chon(t33)
p, q, P, Q, m, d, D = mh.model_["arma"]
print(ar.ten_mo_hinh(mh), "AICc", round(mh.model_["aicc"], 2), ar.kiem_phan_du(mh))
print("ACF phần dư ở trễ 12:", round(float(ar.acf_pacf(mh.model_["residuals"][d + D * m:], 12)["acf"].loc[12]), 2))
tu_dong = ar.auto_arima(t33)
print("auto →", ar.ten_mo_hinh(tu_dong), "AICc", round(tu_dong.model_["aicc"], 2))

## Bước 4 — Khoảng dự báo (mục 4.5)

In [ ]:
bd = ar.du_bao(ar.arima(t33, (0, 1, 0), hang_so=False), 16)
rong = (bd["hi-95"] - bd["lo-95"]).to_numpy()
print("độ rộng log bước 1, 4, 16:", rong[[0, 3, 15]].round(3), "| tỷ lệ so bước 1:", (rong[[3, 15]] / rong[0]).round(2))

## Bước 5 — 366 chuỗi (mục 4.6)

Lần đầu khoảng 6 phút; kết quả lưu vào `du-lieu/cache/`. Cuối cùng: `python lab.py check` trong terminal.

In [ ]:
df = ar.dang_dai(ar.doc_tourism())
kq = ar.backtest_tourism(df, luu=True)
mase = ar.mase_theo_chuoi(kq, df)
print(mase.agg(["mean", "median"]).round(3))
print(ar.so_sanh(mase).round(4).to_string(index=False))